# NFL Prospect Bin Scoring — v2 (NP Embedding Similarity)

**Upgrade from v1:** Instead of counting keyword tokens, we classify **noun phrases** using cosine similarity to bin centroids built from W2V embeddings.

**Pipeline:**
1. Same preprocessing as v1 (phrase stitching, stopwords, lemmatization)
2. Train Word2Vec on full corpus → build embedding space
3. Build **bin centroids** = average W2V vector of anchor seed terms per bin
4. For each player's text: spaCy extracts noun phrases
5. Each NP is embedded (average W2V of its preprocessed tokens) and compared to all 3 centroids
6. NP assigned to nearest bin if similarity > threshold; else 'unclassified'
7. Text between NPs inherits the clause-level assignment as fallback
8. Score = proportion of NP word-coverage per bin

**Why this is better than v1:**
- Catches words not in the dictionary via semantic proximity
- Context-aware: 'reading tackle's weight' → technique (not physical)
- Phrase-level unit is more meaningful than individual tokens
- 'wrist flick sends it 50 yards' → technique without needing 'wrist flick' in any keyword list

In [1]:
import pandas as pd
import numpy as np
import re
import json
from collections import Counter
from itertools import combinations

import spacy
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from IPython.display import display, HTML

nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('omw-1.4',   quiet=True)

# ── W2V config (same as v1) ────────────────────────────────────────────────────
W2V_DIM       = 100
W2V_WINDOW    = 6
W2V_EPOCHS    = 30
W2V_SG        = 1
W2V_MIN_COUNT = 3

# ── NP classifier config ──────────────────────────────────────────────────────
SIM_THRESHOLD  = 0.40   # min cosine sim to assign any bin (else 'none')
                         # lower = more coverage, higher = more precision
CENTROID_FROM  = 'seeds_and_core'  # 'seeds_only' | 'seeds_and_core'

# ── spaCy model ───────────────────────────────────────────────────────────────
# Run: python -m spacy download en_core_web_sm
SPACY_MODEL = 'en_core_web_sm'

# ── Bin palette ───────────────────────────────────────────────────────────────
BIN_NAMES  = ['physical', 'technique', 'character']
BIN_COLOR  = {'physical': '#1565C0', 'technique': '#E65100',
               'character': '#2E7D32', 'none': '#AAAAAA'}
BIN_LABEL  = {'physical': 'Physical', 'technique': 'Technique / Skills',
               'character': 'Character / IQ'}

FILTER_SPECIALISTS = True

print('Setup complete.')

Setup complete.


In [2]:
# Run once to install the spaCy model — safe to re-run (no-ops if already installed)
import subprocess, sys
try:
    import en_core_web_sm
    print('en_core_web_sm already installed')
except ImportError:
    subprocess.run([
        'uv', 'pip', 'install',
        'en-core-web-sm @ https://github.com/explosion/spacy-models/releases/download/'
        'en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl'
    ], check=True, cwd=r'c:\Users\sffra\Downloads\BSE 2025-2026\nfl-draft-nlp')
    print('Installed en_core_web_sm 3.8.0')


en_core_web_sm already installed


## Section 1 — Load spaCy

In [3]:
nlp = spacy.load(SPACY_MODEL)
print(f'spaCy model loaded: {SPACY_MODEL}')
print(f'Pipeline: {nlp.pipe_names}')

spaCy model loaded: en_core_web_sm
Pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


## Section 2 — Data Loading & Preprocessing Config

In [4]:
# ── Curated compound NFL terms ─────────────────────────────────────────────────
_CURATED_RAW = {
    # Trigrams & longer
    'change of direction':      'change_of_direction',
    'low pad level':            'low_pad_level',
    'run after catch':          'run_after_catch',
    'yards after contact':      'yards_after_contact',
    'yards after catch':        'yards_after_catch',
    'run after contact':        'run_after_contact',
    'off the line':             'off_the_line',
    'off the ball':             'off_the_ball',
    'point of attack':          'point_of_attack',
    'get off the line':         'get_off_the_line',
    'read and react':           'read_and_react',
    'inside the box':           'inside_the_box',
    'stack and shed':           'stack_and_shed',
    'pass rush moves':          'pass_rush_moves',
    'tackles for loss':         'tackles_for_loss',
    'tackle for loss':          'tackle_for_loss',
    'setting the edge':         'setting_edge',
    'set the edge':             'setting_edge',
    'turn the corner':          'turn_the_corner',
    'flatten the arc':          'flatten_arc',
    'low to the ground':        'low_to_ground',
    'beat the punch':           'beat_the_punch',
    'block take on':            'block_takeon',
    'block take-on':            'block_takeon',
    'make plays':               'play_making',
    'makes plays':              'play_making',
    'generates pressure':       'generate_pressure',
    'generate pressure':        'generate_pressure',
    'creates pressure':         'generate_pressure',
    'north south':              'north_south',
    'north-south':              'north_south',
    'balanced base':            'balanced_base',
    'wide base':                'wide_base',
    'burst of speed':           'burst_of_speed',
    'arm strength':             'arm_strength',
    'leg drive':                'leg_drive',
    'catch radius':             'catch_radius',
    'catching radius':          'catch_radius',
    'hip tightness':            'hip_tightness',
    'tight hips':               'hip_tightness',
    'high tightness':           'high_tightness',
    'hip bend':                 'hip_bend',
    'bend the edge':            'bend_edge',
    'edge bending':             'bend_edge',
    'lean frame':               'lean_frame',
    'thick frame':              'thick_frame',
    'raw speed':                'raw_speed',
    'raw power':                'raw_power',
    'straight line speed':      'straight_line_speed',
    'straight-line speed':      'straight_line_speed',
    'grip strength':            'grip_strength',
    'wrist flick':              'wrist_flick',
    'change-of-direction':      'change_of_direction',
    'change of direction':      'change_of_direction',
    'release point':            'release_point',
    'arm angles':               'arm_angles',
    'arm angle':                'arm_angles',
    'sees pressure':            'blitz_awareness',
    'reads pressure':           'blitz_awareness',
    'tight window':             'tight_windows',
    'tight windows':            'tight_windows',
    'drive throws':             'drive_throw',
    'cannon arm':               'cannon',
    'body control':             'body_control',
    'quick release':            'quick_delivery',
    'pocket passer':            'pocket_passer',
    'lower half':               'lower_half',
    'lower body':               'lower_body',
    'upper body':               'upper_body',
    'explosive hips':           'explosive_hips',
    'violent club':             'violent_club',
    'club move':                'club_move',
    'spin move':                'spin_move',
    'swim move':                'swim_move',
    'speed rush':               'speed_rush',
    'counter move':             'counter_move',
    'rush move':                'rush_moves',
    'rush moves':               'rush_moves',
    'bull rush':                'bull_rush',
    'bull rusher':              'bull_rusher',
    'bull rushing':             'bull_rush',
    'push pocket':              'push_pocket',
    'attack angle':             'attack_angle',
    'run support':              'run_support',
    'run stopper':              'run_stopper',
    'setting edge':             'setting_edge',
    'run fits':                 'run_fits',
    'run fit':                  'run_fits',
    'run defense':              'run_defense',
    'pursuit angle':            'pursuit_angle',
    'wrap up':                  'wrap_up',
    'sure tackler':             'sure_tackler',
    'downhill trigger':         'downhill_trigger',
    'pass sets':                'pass_sets',
    'arm extension':            'arm_extension',
    'drive blocking':           'drive_blocking',
    'reach blocking':           'reach_blocking',
    'active hands':             'active_hands',
    'hand usage':               'hand_usage',
    'hip sink':                 'hip_sink',
    'pocket presence':          'pocket_presence',
    'edge setter':              'edge_setter',
    'press technique':          'press_technique',
    'off coverage':             'off_coverage',
    'trail technique':          'trail_technique',
    'man to man':               'man_coverage',
    'zone drops':               'zone_drops',
    're route':                 're_route',
    're-route':                 're_route',
    'break on ball':            'break_on_ball',
    'high point':               'high_point',
    'contested catch':          'contested_catch',
    'contested catches':        'contested_catch',
    'post up':                  'post_up',
    'separation quickness':     'separation_quickness',
    'quick delivery':           'quick_delivery',
    'pocket movement':          'pocket_movement',
    'ball placement':           'ball_placement',
    'football intelligence':    'football_intelligence',
    'mental toughness':         'mental_toughness',
    'high effort':              'high_effort',
    'off field':                'off_field',
    'practice habits':          'practice_habits',
    'film room':                'film_room',
    'football character':       'football_character',
    'blitz awareness':          'blitz_awareness',
    'pocket awareness':         'pocket_awareness',
    'eye level':                'eye_level',
    'ultra competitive':        'ultra_competitive',
    'pass rush':                'pass_rush',
    'pass rusher':              'pass_rusher',
    'pass protection':          'pass_protection',
    'pass coverage':            'pass_coverage',
    'pad level':                'pad_level',
    'press coverage':           'press_coverage',
    'man coverage':             'man_coverage',
    'zone coverage':            'zone_coverage',
    'ball skills':              'ball_skills',
    'ball hawk':                'ball_hawk',
    'ball carrier':             'ball_carrier',
    'body control':             'body_control',
    'contact balance':          'contact_balance',
    'closing speed':            'closing_speed',
    'lateral quickness':        'lateral_quickness',
    'quick twitch':             'quick_twitch',
    'high motor':               'high_motor',
    'first step':               'first_step',
    'get off':                  'get_off',
    'hand fighting':            'hand_fighting',
    'hand strength':            'hand_strength',
    'hand placement':           'hand_placement',
    'strong hands':             'strong_hands',
    'soft hands':               'soft_hands',
    'heavy hands':              'heavy_hands',
    'block shedding':           'block_shedding',
    'anchor strength':          'anchor_strength',
    'route running':            'route_running',
    'run blocking':             'run_blocking',
    'open field':               'open_field',
    'red zone':                 'red_zone',
    'ball skills':               'ball_skills',
    'catch radius':              'catch_radius',
    'second level':             'second_level',
    'hip flexibility':          'hip_flexibility',
    'short area':               'short_area',
    'three down':               'three_down',
    'top end':                  'top_end',
    'two gap':                  'two_gap',
    'one gap':                  'one_gap',
    'snap count':               'snap_count',
    'football iq':              'football_iq',
    'film study':               'film_study',
    'work ethic':               'work_ethic',
    'locker room':              'locker_room',
    'decision making':          'decision_making',
    'play making':              'play_making',
    'field vision':             'field_vision',
    'play recognition':         'play_recognition',
    'pre snap':                 'pre_snap',
    'post snap':                'post_snap',
    'game ready':               'game_ready',
    'hard worker':              'hard_worker',
    'functional strength':      'functional_strength',
    'tackling form':            'tackling_form',
    'processing speed':         'processing_speed',
    'eye discipline':           'eye_discipline',
    'spatial awareness':        'spatial_awareness',
    'route progressions':       'route_progressions',
}
CURATED_PHRASE_MAP = dict(
    sorted(_CURATED_RAW.items(), key=lambda x: len(x[0]), reverse=True)
)

# ── NFL-aware stopwords ────────────────────────────────────────────────────────
KEEP_WORDS = {
    'high', 'low', 'heavy', 'light', 'deep', 'short', 'long', 'wide',
    'hard', 'soft', 'strong', 'quick', 'good', 'great',
    'up', 'down', 'off', 'out', 'over', 'through', 'above', 'below',
    'hand', 'hands', 'back', 'field', 'ball'
}
CUSTOM_STOPS = {
    'prospect', 'player', 'players', 'show', 'shows', 'need', 'needs',
    'ability', 'also', 'often', 'must', 'well', 'still', 'use', 'get',
    'make', 'look', 'help', 'time', 'year', 'team', 'game',
    'continue', 'develop', 'development', 'nfl', 'draft', 'college',
    'level', 'type', 'project', 'potential', 'upside',
    'starter', 'backup', 'senior', 'junior', 'cb', 'rb', 'wr', 'qb',
    'scout', 'scouts', 'evaluator', 'evaluators',
    # Position names used as common nouns escape the PROPN filter.
    # "quarterbacks who show their cards", "high safety", "play nickel"
    # are role/alignment labels, not player attributes — strip them.
    'quarterback', 'quarterbacks',
    'cornerback', 'cornerbacks',
    'linebacker', 'linebackers',
    'lineman', 'linemen',
    'nickel', 'dime',
    'defender', 'defenders',
    # League / competition context
    'cfl', 'afl', 'usfl', 'xfl', 'arena',
    'football', 'league', 'season', 'canadian', 'indoor',
}
_base = set(stopwords.words('english'))
NFL_STOPWORDS = (_base - KEEP_WORDS) | CUSTOM_STOPS

print(f'Phrase map: {len(CURATED_PHRASE_MAP)} entries')
print(f'Stopwords : {len(NFL_STOPWORDS)} entries')


Phrase map: 184 entries
Stopwords : 247 entries


In [5]:
df = pd.read_csv('../data/processed/draft_enriched_with_contracts.csv')

keep_cols = ['player_name', 'Pos_Group', 'position', 'grade', 'year',
             'made_it_contract', 'overview', 'strengths', 'weaknesses']
df = df[keep_cols].copy()

for col in ['overview', 'strengths', 'weaknesses']:
    df[col] = df[col].fillna('')

if FILTER_SPECIALISTS:
    n_before = len(df)
    df = df[df['Pos_Group'] != 'SPECIAL'].reset_index(drop=True)
    print(f'Specialists removed: {n_before - len(df)}')

print(f'Players: {len(df)}')
print(f'Positions: {df["Pos_Group"].value_counts().to_dict()}')

Specialists removed: 223
Players: 7173
Positions: {'DB': 1416, 'OL': 1219, 'WR': 1027, 'EDGE': 865, 'RB': 679, 'DT': 615, 'LB': 588, 'TE': 426, 'QB': 334}


## Section 3 — Preprocessing & W2V Training

In [6]:
lemmatizer = WordNetLemmatizer()

def nfl_preprocess(text: str, phrase_map: dict = CURATED_PHRASE_MAP) -> str:
    """NFL-aware preprocessing: normalize → stitch → clean → filter → lemmatize."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    text = re.sub(r'[-\u2013\u2014]', ' ', text)
    for phrase, token in phrase_map.items():
        text = text.replace(phrase, token)
    text = re.sub(r'[^a-z_\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if '_' in t or t not in NFL_STOPWORDS]
    tokens = [t if '_' in t else lemmatizer.lemmatize(t) for t in tokens]
    tokens = [t for t in tokens if len(t) > 1]
    return ' '.join(tokens)

df['all_text']   = df['overview'] + ' ' + df['strengths'] + ' ' + df['weaknesses']
df['all_tokens'] = df['all_text'].apply(nfl_preprocess).apply(str.split)

print(f'Preprocessing complete. Avg tokens per player: {df["all_tokens"].apply(len).mean():.0f}')

Preprocessing complete. Avg tokens per player: 115


In [7]:
sentences = df['all_tokens'].tolist()

w2v = Word2Vec(
    sentences,
    vector_size = W2V_DIM,
    window      = W2V_WINDOW,
    epochs      = W2V_EPOCHS,
    sg          = W2V_SG,
    min_count   = W2V_MIN_COUNT,
    workers     = 4,
    seed        = 42,
)

print(f'W2V vocab size : {len(w2v.wv):,}')
print(f'Vector dims    : {w2v.vector_size}')

W2V vocab size : 7,825
Vector dims    : 100


## Section 4 — Bin Seeds & Centroid Construction

The **centroid** for each bin is the average W2V vector of its anchor seed terms.
This is the prototype that every NP gets compared against via cosine similarity.

We also add core manual terms (unambiguous, high-confidence) to give the centroid
broader coverage without polluting it with noisy terms.

In [ ]:
ANCHOR_SEEDS = {
    'physical': [
        'explosive', 'burst', 'speed', 'acceleration', 'first_step', 'get_off',
        'change_of_direction', 'agility', 'frame', 'size', 'strength',
        'power', 'athletic', 'physical', 'twitch', 'measurable',
    ],
    'technique': [
        'technique', 'footwork', 'leverage', 'pad_level', 'mechanics', 'productive',
        'route_running', 'pass_protection', 'hand_fighting', 'block_shedding',
        'anchor_strength', 'pass_rush', 'blocking', 'tackling', 'coverage',
    ],
    'character': [
        'effort', 'motor', 'high_motor', 'relentless', 'competitive',
        'toughness', 'instinct', 'awareness', 'intelligence', 'football_iq',
        'coachable', 'discipline', 'leadership', 'work_ethic', 'recognition',
    ],
}

# Core manual additions to enrich the centroid (unambiguous terms only)
CENTROID_EXTRAS = {
    'physical':  [
        'height', 'length', 'wingspan', 'arm', 'velocity', 'cannon',
        'speed', 'quickness', 'burst', 'lean', 'thick', 'frame',
        'undersized', 'size', 'weight', 'hands', 'feet',
        'arms', 'lower_half', 'body_control',
        'explosion', 'vertical', 'hip_flexibility', 'lower_half',
        'measurable', 'physicality',
    ],
    'technique': [
        'throw', 'release', 'delivery', 'pocket', 'accuracy', 'timing',
        'separation', 'route_running', 'pass_rush', 'bull_rush', 
        'vision', 'tackling', 'coverage', 'blocking', 'sack', 'ball_skills', 'catch_radius', 'catches'
        'pressure', 'spin_move', 'swim_move', 'run_support',
    ],
    'character': [
        'hustle', 'grit', 'relentless', 'compete', 'smart', 'cerebral',
        'coachable', 'leadership', 'patient', 'composure', 'instinct',
        'mature', 'immature', 'processing', 'dedicated', 'personality',
        'fearless', 'inconsistencies', 'aggressive', 'durable',
        'focus', 'confidence', 'poise',     # mental presence under pressure
        'decisive', 'decisiveness',          # "good decisiveness"
        'competitive',                       # "competitive fuel"
    ],
}

print('Anchor seed vocab coverage:')
for bin_name, seeds in ANCHOR_SEEDS.items():
    in_v  = [s for s in seeds if s in w2v.wv]
    miss  = [s for s in seeds if s not in w2v.wv]
    print(f'  {bin_name:12s}: {len(in_v)}/{len(seeds)} in vocab'
          + (f'  — missing: {miss}' if miss else ''))

Anchor seed vocab coverage:
  physical    : 15/16 in vocab  — missing: ['measurable']
  technique   : 13/15 in vocab  — missing: ['mechanics', 'block_shedding']
  character   : 15/15 in vocab


In [9]:
def build_centroid(terms: list, model: Word2Vec) -> np.ndarray | None:
    """Average W2V vector for a list of terms. Returns None if nothing in vocab."""
    vecs = [model.wv[t] for t in terms if t in model.wv]
    if not vecs:
        return None
    return np.mean(vecs, axis=0)

BIN_CENTROIDS = {}
for bin_name in BIN_NAMES:
    if CENTROID_FROM == 'seeds_only':
        terms = ANCHOR_SEEDS[bin_name]
    else:  # seeds_and_core
        terms = list(set(ANCHOR_SEEDS[bin_name]) | set(CENTROID_EXTRAS[bin_name]))
    centroid = build_centroid(terms, w2v)
    BIN_CENTROIDS[bin_name] = centroid
    n_used = sum(1 for t in terms if t in w2v.wv)
    print(f'  {bin_name:12s}: centroid from {n_used}/{len(terms)} terms in vocab')

# Sanity check: centroids should not be too similar to each other
print('\nInter-centroid cosine similarities (lower = more distinct):')
for b1, b2 in combinations(BIN_NAMES, 2):
    c1, c2 = BIN_CENTROIDS[b1], BIN_CENTROIDS[b2]
    if c1 is not None and c2 is not None:
        sim = float(sk_cosine([c1], [c2])[0][0])
        print(f'  {b1} ↔ {b2}: {sim:.3f}')

  physical    : centroid from 32/36 terms in vocab
  technique   : centroid from 28/32 terms in vocab
  character   : centroid from 33/33 terms in vocab

Inter-centroid cosine similarities (lower = more distinct):
  physical ↔ technique: 0.750
  physical ↔ character: 0.552
  technique ↔ character: 0.647


## Section 5 — NP Extractor + Clause Splitter

For each text:
1. Apply spaCy to get noun phrases (`.noun_chunks`)
2. Split remaining (non-NP) text into clauses at natural boundaries
3. Each unit — NP or clause gap — gets embedded and classified

In [10]:
# ── Pre-compile phrase map as a single regex (replaces 160 sequential str.replace calls) ──
_phrase_keys  = sorted(CURATED_PHRASE_MAP.keys(), key=len, reverse=True)
_phrase_regex = re.compile('|'.join(re.escape(p) for p in _phrase_keys), re.IGNORECASE)

def _apply_phrases(text: str) -> str:
    return _phrase_regex.sub(lambda m: CURATED_PHRASE_MAP[m.group(0).lower()], text)

# ── Pre-normalize centroids once → single matmul replaces 3 sklearn cosine calls ──
_centroid_matrix = np.vstack([BIN_CENTROIDS[b] for b in BIN_NAMES])          # (3, dim)
_norms           = np.linalg.norm(_centroid_matrix, axis=1, keepdims=True)
_centroid_normed = _centroid_matrix / _norms                                   # pre-normalised

# ── Span embedding cache — many NPs repeat across players ─────────────────────
# "long arms", "good speed", "pass rush" appear hundreds of times — embed once.
_embed_cache: dict[str, np.ndarray | None] = {}

def embed_span(text: str) -> np.ndarray | None:
    if text in _embed_cache:
        return _embed_cache[text]

    # Minimal preprocessing: phrases first, then tokenise + lemmatise
    t = text.lower()
    t = re.sub(r'[-\u2013\u2014]', ' ', t)
    t = _apply_phrases(t)
    t = re.sub(r'[^a-z_\s]', ' ', t)
    tokens = t.split()
    tokens = [tok if '_' in tok else lemmatizer.lemmatize(tok)
              for tok in tokens if '_' in tok or tok not in NFL_STOPWORDS]
    tokens = [tok for tok in tokens if len(tok) > 1]

    vecs = [w2v.wv[tok] for tok in tokens if tok in w2v.wv]
    result = np.mean(vecs, axis=0).astype(np.float32) if vecs else None
    _embed_cache[text] = result
    return result


def classify_span(text: str, threshold: float = SIM_THRESHOLD) -> dict:
    vec = embed_span(text)
    if vec is None:
        return {'bin': 'none', 'sims': {}, 'vec_found': False}

    norm = np.linalg.norm(vec)
    if norm == 0:
        return {'bin': 'none', 'sims': {}, 'vec_found': False}

    # Single matrix-vector multiply → all 3 similarities at once
    sims_arr = _centroid_normed @ (vec / norm)          # shape (3,)
    sims     = {b: float(sims_arr[i]) for i, b in enumerate(BIN_NAMES)}

    best_idx = int(np.argmax(sims_arr))
    assigned = BIN_NAMES[best_idx] if sims_arr[best_idx] >= threshold else 'none'
    return {'bin': assigned, 'sims': sims, 'vec_found': True}


# ── Sanity check ───────────────────────────────────────────────────────────────
_embed_cache.clear()   # fresh cache for the test
test_spans = [
    ('long arms',              'physical'),
    ('stab move',              'technique'),
    ('top-end speed',          'physical'),
    ('proper pursuit angles',  'technique'),
    ('his football iq',        'character'),
    ('wrist flick',            'technique'),
    ('reading tackle weight',  'technique'),
    ('high motor guy',         'character'),
]
print(f'{"Span":<35} {"Expected":<12} {"Got":<12} {"Sims"}')
print('-' * 80)
for span, expected in test_spans:
    r = classify_span(span)
    sims_str = '  '.join(f'{b[0].upper()}:{v:.2f}' for b, v in r['sims'].items())
    mark = '✓' if r['bin'] == expected else '✗'
    print(f'{mark} {span:<33} {expected:<12} {r["bin"]:<12} {sims_str}')

print(f'\nCache size after test: {len(_embed_cache)} entries')


Span                                Expected     Got          Sims
--------------------------------------------------------------------------------
✓ long arms                         physical     physical     P:0.68  T:0.50  C:0.31
✓ stab move                         technique    technique    P:0.45  T:0.52  C:0.27
✓ top-end speed                     physical     physical     P:0.64  T:0.45  C:0.43
✓ proper pursuit angles             technique    technique    P:0.39  T:0.53  C:0.37
✓ his football iq                   character    character    P:0.28  T:0.33  C:0.58
✓ wrist flick                       technique    technique    P:0.48  T:0.49  C:0.30
✗ reading tackle weight             technique    physical     P:0.54  T:0.53  C:0.47
✓ high motor guy                    character    character    P:0.38  T:0.33  C:0.58

Cache size after test: 8 entries


## Section 6 — Annotate Text: NPs + Clause Gaps

In [11]:
# ── Coordinated NP splitter ────────────────────────────────────────────────────
# Splits "good technique and unwavering energy" → ["good technique", "unwavering energy"]
# so each part gets its own classification rather than averaging across bins.
# Only splits on bare "and"/"or" — curated phrases like "read_and_react" are already
# stitched into single tokens before this runs.
_SKIP_ENT_TYPES = {"ORG", "GPE", "NORP", "FAC", "LOC"}
_COORD_CONJ = re.compile(r'\s+(?:and|or)\s+', re.IGNORECASE)

def _split_coord_np(text: str, start_char: int) -> list[tuple[str, int, int]]:
    """
    Split a coordinated NP into sub-spans with character offsets.
    'good technique and unwavering energy' → [('good technique', s, e), ('unwavering energy', s2, e2)]
    Returns the original single-element list if no conjunction found.
    """
    # Don't split if the text contains a curated phrase (already stitched)
    if '_' in _apply_phrases(text.lower()):
        return [(text, start_char, start_char + len(text))]

    parts = []
    pos   = 0
    for m in _COORD_CONJ.finditer(text):
        part = text[pos:m.start()]
        if part.strip():
            parts.append((part, start_char + pos, start_char + m.start()))
        pos = m.end()
    tail = text[pos:]
    if tail.strip():
        parts.append((tail, start_char + pos, start_char + len(text)))

    return parts if len(parts) > 1 else [(text, start_char, start_char + len(text))]


def annotate_doc(doc) -> list[dict]:
    """
    Annotate a pre-computed spaCy doc at NP level.
    Accepts a doc object so the caller can batch nlp.pipe() instead of
    calling nlp() per player (10x speedup).
    Coordinated NPs ('technique and energy') are split before classification.
    """
    raw_text = doc.text
    if not raw_text.strip():
        return []

    spans = []

    for chunk in doc.noun_chunks:
        if chunk.root.pos_ == 'PROPN':   # skip player/team names
            continue
        if any(tok.i in {t.i for ent in doc.ents if ent.label_ in _SKIP_ENT_TYPES for t in ent} for tok in chunk):
            continue  # skip org/location named entities (e.g. CFL, XFL)

        sub_spans = _split_coord_np(chunk.text, chunk.start_char)
        for sub_text, sub_start, sub_end in sub_spans:
            result  = classify_span(sub_text)
            n_words = sum(1 for w in sub_text.split() if w.strip())
            spans.append({
                'text':    sub_text,
                'start':   sub_start,
                'end':     sub_end,
                'kind':    'np',
                'bin':     result['bin'],
                'sims':    result['sims'],
                'n_words': n_words,
            })

    return sorted(spans, key=lambda s: s['start'])


def annotate_text(raw_text: str) -> list[dict]:
    """Single-text convenience wrapper (used for spot-checks only)."""
    if not isinstance(raw_text, str) or not raw_text.strip():
        return []
    return annotate_doc(nlp(raw_text))


# Quick tests
tests = [
    "Has the long arms to develop an effective stab move as a pass rush option.",
    "plays with good technique and unwavering energy",
    "Taylor-Demerson might not have the highly coveted measurables that teams will gravitate toward.",
]
for sent in tests:
    ann = annotate_text(sent)
    print(f'\n"{sent}"')
    for s in ann:
        if s['bin'] != 'none':
            print(f'  [{s["bin"]:10}]  "{s["text"]}"')



"Has the long arms to develop an effective stab move as a pass rush option."
  [physical  ]  "the long arms"
  [technique ]  "an effective stab move"
  [technique ]  "a pass rush option"

"plays with good technique and unwavering energy"
  [technique ]  "good technique"
  [character ]  "unwavering energy"

"Taylor-Demerson might not have the highly coveted measurables that teams will gravitate toward."
  [character ]  "the highly coveted measurables"


## Section 7 — Score Players

In [12]:
def score_from_annotations(annotations: list[dict]) -> dict:
    """
    Convert a list of annotated spans into bin percentage scores.

    Each span has:
      'bin'     : 'physical' | 'technique' | 'character' | 'none'
      'n_words' : number of words in the span

    Returns a dict with physical_pct, technique_pct, character_pct,
    total_classified_words (words assigned to any bin).
    """
    counts = {b: 0 for b in BIN_NAMES}
    for span in annotations:
        b = span.get('bin', 'none')
        if b in counts:
            counts[b] += span.get('n_words', 0)

    total = sum(counts.values())
    if total == 0:
        return {f'{b}_pct': 0.0 for b in BIN_NAMES} | {'total_classified_words': 0}

    return {f'{b}_pct': counts[b] / total for b in BIN_NAMES} | {'total_classified_words': total}


# Quick sanity check
_test_ann = [
    {'bin': 'physical',  'n_words': 3},
    {'bin': 'technique', 'n_words': 2},
    {'bin': 'none',      'n_words': 5},
    {'bin': 'character', 'n_words': 1},
]
_s = score_from_annotations(_test_ann)
assert abs(_s['physical_pct']  - 0.5)  < 1e-6
assert abs(_s['technique_pct'] - 1/3)  < 1e-6
assert abs(_s['character_pct'] - 1/6)  < 1e-6
assert _s['total_classified_words'] == 6
print('score_from_annotations OK')

score_from_annotations OK


In [ ]:
import time

BATCH_SIZE = 128

# ── Step 1: spaCy batch parse ──────────────────────────────────────────────────
text_records = []
for idx, row in df.iterrows():
    combined = ' '.join(filter(None, [
        str(row.get('overview',   '') or ''),
        str(row.get('strengths',  '') or ''),
        str(row.get('weaknesses', '') or ''),
    ]))
    text_records.append((idx, '',     combined))
    text_records.append((idx, 'str_', str(row.get('strengths',  '') or '')))
    text_records.append((idx, 'wk_',  str(row.get('weaknesses', '') or '')))

texts = [t for _, _, t in text_records]
t0    = time.time()
print(f'spaCy parsing {len(texts)} texts...')
docs  = list(nlp.pipe(texts, batch_size=BATCH_SIZE, disable=[]))
print(f'  done in {time.time()-t0:.1f}s')

# ── Step 2: Extract spans (with coord-NP splitting + name filtering) ───────────
print('Extracting spans...')

all_span_records = []
doc_span_slices  = []

def _tokens_from_text(text: str) -> list[str]:
    """Text-based token extraction for coord-split sub-spans."""
    raw      = text.lower()
    raw      = re.sub(r'[-\u2013\u2014]', ' ', raw)
    stitched = _apply_phrases(raw)
    stitched = re.sub(r'[^a-z_\s]', ' ', stitched)
    phrases  = [t for t in stitched.split() if '_' in t]
    tokens   = [t for t in stitched.split()
                if not '_' in t and t not in NFL_STOPWORDS and len(t) > 1]
    tokens   = [lemmatizer.lemmatize(t) for t in tokens]
    return phrases + tokens

def _tokens_from_spacy_span(chunk) -> list[str]:
    """Use spaCy lemmas + phrase stitching for full chunks. No WordNet calls."""
    raw      = chunk.text.lower()
    raw      = re.sub(r'[-\u2013\u2014]', ' ', raw)
    stitched = _apply_phrases(raw)
    stitched = re.sub(r'[^a-z_\s]', ' ', stitched)
    phrases  = [t for t in stitched.split() if '_' in t]
    lemmas   = [t.lemma_.lower() for t in chunk
                if not t.is_punct and not t.is_space
                and len(t.lemma_) > 1
                and t.lemma_.lower() not in NFL_STOPWORDS]
    return phrases + lemmas

for rec_i, ((idx, prefix, _), doc) in enumerate(zip(text_records, docs)):
    spans_here = []

    for chunk in doc.noun_chunks:
        if chunk.root.pos_ == 'PROPN':   # skip standalone player/team names
            continue
        _ent_idxs = {t.i for ent in doc.ents if ent.label_ in _SKIP_ENT_TYPES for t in ent}
        if any(tok.i in _ent_idxs for tok in chunk):
            continue  # skip org/location named entities

        sub_spans = _split_coord_np(chunk.text, chunk.start_char)

        if len(sub_spans) == 1:
            # No split — use spaCy lemmas (faster, no WordNet)
            tokens  = _tokens_from_spacy_span(chunk)
            n_words = sum(1 for t in chunk if not t.is_punct and not t.is_space)
            spans_here.append({
                'text':    chunk.text,
                'tokens':  tokens,
                'start':   chunk.start_char,
                'end':     chunk.end_char,
                'kind':    'np',
                'n_words': n_words,
            })
        else:
            # Coordinated NP — classify each part separately
            for sub_text, sub_start, sub_end in sub_spans:
                tokens  = _tokens_from_text(sub_text)
                n_words = len(sub_text.split())
                spans_here.append({
                    'text':    sub_text,
                    'tokens':  tokens,
                    'start':   sub_start,
                    'end':     sub_end,
                    'kind':    'np',
                    'n_words': n_words,
                })

    start = len(all_span_records)
    all_span_records.extend([(rec_i, s) for s in spans_here])
    doc_span_slices.append((rec_i, idx, prefix, start, len(all_span_records)))

print(f'  {len(all_span_records)} total spans extracted')

# ── Step 3: Deduplicate by token tuple → embed unique sets only ────────────────
print('Embedding unique spans...')
token_key_to_vec: dict = {}

for _, span in all_span_records:
    key = tuple(span['tokens'])
    if key not in token_key_to_vec:
        vecs = [w2v.wv[t] for t in span['tokens'] if t in w2v.wv]
        token_key_to_vec[key] = np.mean(vecs, axis=0).astype(np.float32) if vecs else None

print(f'  {len(token_key_to_vec)} unique token sets')

# ── Step 4: One matrix multiply for all similarities ──────────────────────────
keys_with_vec = [(k, v) for k, v in token_key_to_vec.items() if v is not None]

key_to_result: dict = {}
if keys_with_vec:
    keys_list, vecs_list = zip(*keys_with_vec)
    span_mat   = np.vstack(vecs_list)
    norms      = np.linalg.norm(span_mat, axis=1, keepdims=True)
    span_normd = span_mat / np.maximum(norms, 1e-8)
    sim_mat    = span_normd @ _centroid_normed.T                  # (n, 3)

    best_bins  = np.argmax(sim_mat, axis=1)
    best_sims  = sim_mat[np.arange(len(keys_list)), best_bins]

    for i, key in enumerate(keys_list):
        assigned = BIN_NAMES[best_bins[i]] if best_sims[i] >= SIM_THRESHOLD else 'none'
        key_to_result[key] = {
            'bin':  assigned,
            'sims': {b: float(sim_mat[i, j]) for j, b in enumerate(BIN_NAMES)},
        }

# ── Step 5: Assemble df_result ─────────────────────────────────────────────────
print('Assembling scores...')
player_scores = {}

for rec_i, idx, prefix, span_start, span_end in doc_span_slices:
    annotations = []
    for _, span in all_span_records[span_start:span_end]:
        key    = tuple(span['tokens'])
        result = key_to_result.get(key, {'bin': 'none', 'sims': {}})
        annotations.append({**span, **result})

    score = score_from_annotations(sorted(annotations, key=lambda x: x['start']))
    if idx not in player_scores:
        player_scores[idx] = {}
    player_scores[idx].update({f'{prefix}{k}': v for k, v in score.items()})

ID_COLS   = ['player_name', 'Pos_Group', 'position', 'grade', 'year', 'made_it_contract']
scores_df = pd.DataFrame([player_scores[i] for i in df.index])
df_result = pd.concat([df[ID_COLS].reset_index(drop=True), scores_df], axis=1)

print(f'\n✓ Done in {time.time()-t0:.1f}s  |  {len(df_result)} players scored')
display_cols = ID_COLS + ['physical_pct', 'technique_pct', 'character_pct', 'total_classified_words']
print(df_result[display_cols].head(3).to_string(index=False))


spaCy parsing 21519 texts...


## Section 8 — Validation & Spot Checks

In [ ]:
# ── Coverage diagnostics ──────────────────────────────────────────────────────
zero = df_result['total_classified_words'] == 0
low  = df_result['total_classified_words'].between(1, 10)
print(f'Players with 0 classified words : {zero.sum()}')
print(f'Players with 1-10 classified     : {low.sum()}')
print(f'Players with >10 classified      : {(df_result["total_classified_words"] > 10).sum()}')

print('\nMean bin % by position group:')
pos_summary = (
    df_result[df_result['total_classified_words'] > 0]
    .groupby('Pos_Group')[['physical_pct', 'technique_pct', 'character_pct']]
    .mean()
    .multiply(100)
    .round(1)
    .rename(columns={'physical_pct': 'Phys%', 'technique_pct': 'Tech%', 'character_pct': 'Char%'})
    .sort_values('Phys%', ascending=False)
)
print(pos_summary.to_string())

In [ ]:
def player_report(name: str) -> None:
    """Print a scoring breakdown + NP annotations for one player."""
    matches = df_result[df_result['player_name'].str.lower().str.contains(name.lower())]
    if matches.empty:
        print(f'Not found: {name}')
        return
    row    = matches.iloc[0]
    df_row = df[df['player_name'] == row['player_name']].iloc[0]

    print('=' * 60)
    print(f'  {row["player_name"]}  |  {row["position"]}  |  Grade: {row["grade"]}')
    print('=' * 60)
    for b in BIN_NAMES:
        pct = row[f'{b}_pct'] * 100
        bar = '█' * int(pct / 5)
        print(f'  {b:12s} {pct:5.1f}%  {bar}')
    print(f'  Total classified words: {row["total_classified_words"]}')

    print('\nNP-level annotations (strengths):')
    anns = annotate_text(df_row['strengths'])
    for a in anns:
        if a['bin'] != 'none' and a['kind'] == 'np':
            sims_str = '  '.join(f"{b[0].upper()}:{v:.2f}" for b, v in a['sims'].items())
            print(f'  [{a["bin"]:10}]  "{a["text"]}"  ({sims_str})')


# Spot check a few players
for name in df_result['player_name'].sample(3, random_state=7).tolist():
    player_report(name)
    print()

## Section 9 — HTML Colorizer

Each NP is colored by its assigned bin. Gaps between NPs are colored at lower opacity.
Hover over any span to see the raw cosine similarity scores.

In [ ]:
def colorize_annotated(raw_text: str, annotations: list[dict]) -> str:
    """Render annotated text as HTML with NP-level coloring."""
    if not annotations:
        return raw_text

    parts    = []
    prev_end = 0

    for span in sorted(annotations, key=lambda s: s['start']):
        # Emit any unannotated text before this span
        if span['start'] > prev_end:
            parts.append(raw_text[prev_end:span['start']])

        color   = BIN_COLOR[span['bin']]
        opacity = '1.0' if span['kind'] == 'np' else '0.6'
        weight  = 'bold' if span['kind'] == 'np' and span['bin'] != 'none' else 'normal'

        if span['bin'] != 'none':
            sims_tip = '  |  '.join(
                f"{BIN_LABEL[b]}: {v:.2f}"
                for b, v in sorted(span['sims'].items(), key=lambda x: -x[1])
            )
            parts.append(
                f'<span style="color:{color};font-weight:{weight};opacity:{opacity}"'
                f' title="{sims_tip}">{raw_text[span["start"]:span["end"]]}</span>'
            )
        else:
            parts.append(raw_text[span['start']:span['end']])

        prev_end = span['end']

    if prev_end < len(raw_text):
        parts.append(raw_text[prev_end:])

    return ''.join(parts)


def player_card_html(player_name: str) -> str:
    """Build full HTML card for one player with NP-level coloring."""
    res_row = df_result[df_result['player_name'] == player_name]
    raw_row = df[df['player_name'] == player_name]
    if res_row.empty or raw_row.empty:
        return f'<p>Not found: {player_name}</p>'

    r   = res_row.iloc[0]
    raw = raw_row.iloc[0]

    p_pct = r['physical_pct']  * 100
    t_pct = r['technique_pct'] * 100
    c_pct = r['character_pct'] * 100

    def section(title, text):
        if not isinstance(text, str) or not text.strip():
            return ''
        ann     = annotate_text(text)
        colored = colorize_annotated(text, ann)
        return (
            f'<p style="margin:10px 0 3px 0;font-size:11px;font-weight:700;'
            f'text-transform:uppercase;letter-spacing:.06em;color:#888">{title}</p>'
            f'<p style="margin:0 0 12px 0;font-size:13px;line-height:1.8">{colored}</p>'
        )

    bar = ''.join(
        f'<div style="display:inline-block;width:{pct:.1f}%;background:{BIN_COLOR[b]};height:12px"'
        f' title="{BIN_LABEL[b]}: {pct:.1f}%"></div>'
        for b, pct in [('physical', p_pct), ('technique', t_pct), ('character', c_pct)]
    )
    legend = ''.join(
        f'<span style="display:inline-block;margin-right:16px;font-size:12px">'
        f'<span style="background:{BIN_COLOR[b]};color:#fff;padding:2px 7px;'
        f'border-radius:3px;font-size:11px">{BIN_LABEL[b]}</span>&nbsp;{pct:.1f}%</span>'
        for b, pct in [('physical', p_pct), ('technique', t_pct), ('character', c_pct)]
    )
    grade = f'{r["grade"]:.2f}' if isinstance(r['grade'], float) else r['grade']

    return f"""
    <div style="border:1px solid #444;border-radius:8px;padding:18px 22px;
                margin:10px 0;font-family:'Segoe UI',Arial,sans-serif;max-width:900px">
      <div style="display:flex;justify-content:space-between;align-items:baseline;
                  border-bottom:2px solid #444;padding-bottom:8px;margin-bottom:14px">
        <span style="font-size:17px;font-weight:700">{player_name}</span>
        <span style="font-size:12px;color:#999">{r['position']} &nbsp;·&nbsp;
              {int(r['year'])} Draft &nbsp;·&nbsp; Grade&nbsp;{grade}</span>
      </div>
      {section('Overview',   raw.get('overview',   ''))}
      {section('Strengths',  raw.get('strengths',  ''))}
      {section('Weaknesses', raw.get('weaknesses', ''))}
      <div style="margin-top:14px;border-top:1px solid #444;padding-top:10px">
        <div style="width:100%;border-radius:4px;overflow:hidden;margin-bottom:7px">{bar}</div>
        {legend}
        <span style="font-size:11px;color:#666;margin-left:12px">
          ({int(r['total_classified_words'])} classified words)
        </span>
      </div>
    </div>"""

In [ ]:
# ── Demo: two players per position group (2013 vs 2025, grade >= 6.0) ──────────
EARLY_POOL = [2013, 2014]
LATE_POOL  = [2024, 2025]
MIN_GRADE  = 6.0

def pick_pair(grp):
    grp = grp[grp['grade'] >= MIN_GRADE]
    early = grp[grp['year'].isin(EARLY_POOL)]
    late  = grp[grp['year'].isin(LATE_POOL)]
    if early.empty or late.empty:
        return None, None
    best_diff, best_e, best_l = -1, None, None
    for _, e in early.iterrows():
        for _, l in late.iterrows():
            diff = abs(float(e['physical_pct']) - float(l['physical_pct']))
            if diff > best_diff:
                best_diff, best_e, best_l = diff, e, l
    return best_e, best_l

html_out = []
skipped  = []

for pos_group, grp in df_result.groupby('Pos_Group'):
    early, late = pick_pair(grp)
    if early is None:
        skipped.append(pos_group)
        continue
    html_out.append(
        f'<h2 style="font-family:sans-serif;margin:36px 0 4px 0;font-size:15px;'
        f'font-weight:700;color:#ccc;border-bottom:2px solid #444;padding-bottom:4px">'
        f'{pos_group}</h2>'
    )
    html_out.append(player_card_html(early['player_name']))
    html_out.append(player_card_html(late['player_name']))

if skipped:
    html_out.append(f'<p style="color:#888;font-size:12px">Skipped: {", ".join(skipped)}</p>')

display(HTML('\n'.join(html_out)))

## Section 10 — Threshold Sensitivity

The `SIM_THRESHOLD` controls how much of the text gets classified. Tune this here.

In [ ]:
# See how coverage changes with different thresholds on a sample player
sample_name = df_result[df_result['total_classified_words'] > 20]['player_name'].iloc[5]
sample_text = df[df['player_name'] == sample_name]['strengths'].iloc[0]

print(f'Player: {sample_name}')
print(f'Text: {sample_text[:200]}...')
print()

for thresh in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35]:
    ann   = annotate_text(sample_text)
    # Re-apply threshold manually
    for s in ann:
        if s['sims']:
            best = max(s['sims'], key=s['sims'].get)
            s['bin'] = best if s['sims'][best] >= thresh else 'none'
    scores = score_from_annotations(ann)
    pcts   = '  '.join(f"{b[0].upper()}:{scores[f'{b}_pct']*100:.0f}%" for b in BIN_NAMES)
    print(f'  thresh={thresh:.2f}  classified={scores["total_classified_words"]:3}  {pcts}')

## Section 11 — Export

In [ ]:
out_path = '../data/processed/bin_scores_v2_NP.csv'
df_result.to_csv(out_path, index=False)
print(f'Saved → {out_path}')
print(f'Shape: {df_result.shape}')

## Section 12 — Trend Plots: Era Comparison

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

BIN_COLS   = ['physical_pct', 'technique_pct', 'character_pct']
BIN_TITLES = ['Physical', 'Technique / Skills', 'Character / IQ']
BIN_COLORS = ['#1565C0', '#E65100', '#2E7D32']

# ── 1. Filter ─────────────────────────────────────────────────────────────────
mask = (df_result['year'] >= 2014) & (df_result['year'] <= 2025) & (df_result['grade'] >= 6.0)
df_filtered = df_result[mask].copy()

# ── 2. Define eras ────────────────────────────────────────────────────────────
ERA_CUTS   = [2017, 2021]
ERA_LABELS = [f'2014–{ERA_CUTS[0]}', f'{ERA_CUTS[0]+1}–{ERA_CUTS[1]}', f'{ERA_CUTS[1]+1}+']

def assign_era(year):
    if year <= ERA_CUTS[0]:   return ERA_LABELS[0]
    elif year <= ERA_CUTS[1]: return ERA_LABELS[1]
    else:                     return ERA_LABELS[2]

df_filtered['era'] = df_filtered['year'].apply(assign_era)

era_pos = (
    df_filtered
    .groupby(['era', 'Pos_Group'])[BIN_COLS]
    .mean()
    .multiply(100)
    .round(1)
    .reset_index()
)
era_order = {label: i for i, label in enumerate(ERA_LABELS)}
era_pos['era_order'] = era_pos['era'].map(era_order)
era_pos = era_pos.sort_values(['era_order', 'Pos_Group']).reset_index(drop=True)

# ── 3. Plot ───────────────────────────────────────────────────────────────────
pos_groups = sorted(era_pos['Pos_Group'].unique())
CMAP       = plt.get_cmap('tab10')
pg_colors  = {pg: CMAP(i) for i, pg in enumerate(pos_groups)}

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
global_max = era_pos[BIN_COLS].max().max()

for ax, col, title, accent in zip(axes, BIN_COLS, BIN_TITLES, BIN_COLORS):
    for pg in pos_groups:
        sub = era_pos[era_pos['Pos_Group'] == pg].sort_values('era_order')
        if sub.empty:
            continue
        ax.plot(sub['era'], sub[col], marker='o', linewidth=2, markersize=7,
                color=pg_colors[pg], label=pg)
        last = sub.iloc[-1]
        ax.annotate(pg, xy=(last['era'], last[col]),
                    xytext=(4, 0), textcoords='offset points',
                    fontsize=7, va='center', color=pg_colors[pg])

    ax.set_title(title, fontsize=14, fontweight='bold', color=accent, pad=10)
    ax.set_xlabel('Era', fontsize=10)
    if ax == axes[0]:
        ax.set_ylabel('Mean % of scouting language', fontsize=10)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
    ax.set_xticks(range(len(ERA_LABELS)))
    ax.set_xticklabels(ERA_LABELS, rotation=10, ha='right', fontsize=9)
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_ylim(0, global_max * 1.15)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Position Group', loc='lower center',
           ncol=len(pos_groups), fontsize=9, title_fontsize=10,
           frameon=True, bbox_to_anchor=(0.5, -0.08))
plt.suptitle('NFL Scouting Language Composition by Era — v2 NP (2014–2025 | Grade ≥ 6.0)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

chart_path = '../data/processed/bin_trends_by_era_v2.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved → {chart_path}')

## Section 13 — Trend Plots: Yearly Rolling Average (All Positions)

In [ ]:
BIN_COLS   = ['physical_pct', 'technique_pct', 'character_pct']
BIN_TITLES = ['Physical', 'Technique / Skills', 'Character / IQ']
BIN_COLORS = ['#1565C0', '#E65100', '#2E7D32']

# ── 1. Filter ─────────────────────────────────────────────────────────────────
mask = (
    (df_result['year']  >= 2011) &
    (df_result['year']  <= 2025) &
    (df_result['grade'] >= 5.5)  &
    (df_result['total_classified_words'] > 0)
)
df_yearly = df_result[mask].copy()

# ── 2. Aggregate per year × position ──────────────────────────────────────────
yearly_stats = (
    df_yearly
    .groupby(['year', 'Pos_Group'])[BIN_COLS]
    .mean()
    .multiply(100)
    .reset_index()
)

# ── 3. 3-year rolling average per position group ──────────────────────────────
yearly_stats = yearly_stats.sort_values(['Pos_Group', 'year'])
for col in BIN_COLS:
    yearly_stats[f'{col}_smooth'] = (
        yearly_stats.groupby('Pos_Group')[col]
        .transform(lambda x: x.rolling(window=3, center=True, min_periods=1).mean())
    )

# ── 4. Plot ───────────────────────────────────────────────────────────────────
pos_groups = sorted(yearly_stats['Pos_Group'].unique())
CMAP       = plt.get_cmap('tab10')
pg_colors  = {pg: CMAP(i) for i, pg in enumerate(pos_groups)}

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
smooth_cols = [f'{c}_smooth' for c in BIN_COLS]
global_max  = yearly_stats[smooth_cols].max().max()

for ax, col, title, accent in zip(axes, BIN_COLS, BIN_TITLES, BIN_COLORS):
    smooth_col = f'{col}_smooth'
    for pg in pos_groups:
        sub = yearly_stats[yearly_stats['Pos_Group'] == pg]
        ax.plot(sub['year'], sub[smooth_col],
                linewidth=3, alpha=0.9, color=pg_colors[pg], label=pg)
        ax.scatter(sub['year'], sub[col],
                   color=pg_colors[pg], s=15, alpha=0.2)
        last = sub.iloc[-1]
        ax.annotate(pg, xy=(last['year'], last[smooth_col]),
                    xytext=(5, 0), textcoords='offset points',
                    fontsize=7, va='center', color=pg_colors[pg], fontweight='bold')

    ax.set_title(title, fontsize=14, fontweight='bold', color=accent, pad=15)
    ax.set_xlabel('Draft Year', fontsize=10)
    if ax == axes[0]:
        ax.set_ylabel('Mean % of scouting language\n(3yr smoothed)', fontsize=10)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
    ax.set_xticks(range(2011, 2026, 2))
    ax.grid(True, axis='y', linestyle='--', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_ylim(0, global_max * 1.15)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Position Group', loc='lower center',
           ncol=len(pos_groups), bbox_to_anchor=(0.5, -0.1), frameon=True)
plt.suptitle('Temporal Evolution of Scouting Traits — v2 NP (2011–2025 | Grade ≥ 5.5)',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()

chart_path = '../data/processed/bin_trends_yearly_v2.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved → {chart_path}')

## Section 14 — Trend Plots: Per-Position Group Breakdown (Combined / Strengths / Weaknesses)

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
YEAR_RANGE    = (2011, 2025)
MIN_GRADE     = 0
SMOOTH_WINDOW = 3
SAVE_CHARTS   = True

# (column prefix, total_words col, panel title)
SOURCES = [
    ('',     'total_classified_words',     'Combined'),
    ('str_', 'str_total_classified_words', 'Strengths'),
    ('wk_',  'wk_total_classified_words',  'Weaknesses'),
]

# (pct column suffix, display label, color)
BINS = [
    ('physical_pct',  'Physical',           '#1565C0'),
    ('technique_pct', 'Technique / Skills', '#E65100'),
    ('character_pct', 'Character / IQ',     '#2E7D32'),
]

# ── Loop over position groups ─────────────────────────────────────────────────
pos_groups = sorted(df_result['Pos_Group'].dropna().unique())

for pg in pos_groups:
    base_mask = (
        (df_result['year']      >= YEAR_RANGE[0]) &
        (df_result['year']      <= YEAR_RANGE[1]) &
        (df_result['grade']     >= MIN_GRADE)      &
        (df_result['Pos_Group'] == pg)
    )
    df_pg = df_result[base_mask].copy()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    fig.suptitle(f'{pg} — Scouting Language Composition Over Time (v2 NP)',
                 fontsize=15, fontweight='bold', y=1.03)

    global_max = 0

    for ax, (prefix, total_col, panel_title) in zip(axes, SOURCES):
        df_src   = df_pg[df_pg[total_col] > 0].copy()
        bin_cols = [f'{prefix}{suf}' for suf, _, _ in BINS]

        yearly = (
            df_src
            .groupby('year')[bin_cols]
            .mean()
            .multiply(100)
            .reset_index()
            .sort_values('year')
        )

        for col in bin_cols:
            yearly[f'{col}_smooth'] = (
                yearly[col]
                .rolling(window=SMOOTH_WINDOW, center=True, min_periods=1)
                .mean()
            )

        smooth_cols = [f'{c}_smooth' for c in bin_cols]
        panel_max   = yearly[smooth_cols].max().max()
        if not np.isnan(panel_max):
            global_max = max(global_max, panel_max)

        for col_suf, bin_label, color in BINS:
            col    = f'{prefix}{col_suf}'
            smooth = f'{col}_smooth'
            if col not in yearly.columns or yearly.empty:
                continue
            ax.plot(yearly['year'], yearly[smooth],
                    linewidth=2.5, color=color, label=bin_label)
            ax.scatter(yearly['year'], yearly[col],
                       color=color, s=12, alpha=0.2)

        ax.set_title(panel_title, fontsize=12, fontweight='bold', pad=8)
        ax.set_xlabel('Draft Year', fontsize=9)
        if ax == axes[0]:
            ax.set_ylabel('Mean % of scouting language\n(3yr smoothed)', fontsize=9)
        ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
        ax.set_xticks(range(YEAR_RANGE[0], YEAR_RANGE[1] + 1, 2))
        ax.tick_params(axis='x', rotation=30, labelsize=8)
        ax.grid(True, axis='y', linestyle='--', alpha=0.3)
        ax.spines[['top', 'right']].set_visible(False)

    y_top = max(global_max * 1.18, 10)
    for ax in axes:
        ax.set_ylim(0, y_top)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, title='Bin', loc='lower center',
               ncol=3, bbox_to_anchor=(0.5, -0.12),
               fontsize=9, title_fontsize=10, frameon=True)
    plt.tight_layout()

    if SAVE_CHARTS:
        out = f'../data/processed/pos_breakdown_{pg.lower()}_v2.png'
        plt.savefig(out, dpi=150, bbox_inches='tight')
        print(f'Saved → {out}')

    plt.show()